In [1]:
import unicodedata
import json, copy, os, re

import pickle
from collections import Counter

import pandas as pd
from tqdm import tqdm

In [2]:
import sys
sys.path.append(r"/Users/nirajh/Code/Language_adaptation/3_unigram_tok")
import tokenizer_edge_cases as ec

In [3]:
from tokenizers import Tokenizer, decoders, pre_tokenizers
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from datasets import load_dataset

#### Loading the tokenizer and modifying its regex pattern

In [4]:
model_dir = "/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"

# Loading the tokenizer
tok = AutoTokenizer.from_pretrained(model_dir)

## Changing the regex pattern in the pre tokenizer, so that kannada words are split correctly
tj = json.loads(tok.backend_tokenizer.to_str())        # Getting the tokenizer's vocab, merges, special tokens, and pre-processing configs
split = tj["pre_tokenizer"]["pretokenizers"][0]        # Fetches the first pre-tokenizer (they are usually chained together under a Sequence)
old = split["pattern"]["Regex"]
new = old.replace(r"[^\r\n\p{L}\p{N}]?\p{L}+", r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+")
assert new != old, "pattern not found; print(old) and adjust"
split["pattern"]["Regex"] = new

# Checking its output
kn_tok = Tokenizer.from_str(json.dumps(tj))
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in kn_tok.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ್ಕಾರ', ' ಕನ್ನಡ', ' ಭಾಷೆಯಲ್ಲಿ']


#### Loading the Kannada dataset

In [5]:
dataset_dir = "/Users/nirajh/Code/Language_adaptation/sangraha/train_split"
print(f"Loading the kannada dataset from the following path:\n{dataset_dir}\n")

# Load the dataset 
kn_train = load_dataset(
    "ai4bharat/sangraha",
    data_dir="verified/kan",
    cache_dir=dataset_dir,
    download_mode="reuse_cache_if_exists"
)

Loading the kannada dataset from the following path:
/Users/nirajh/Code/Language_adaptation/sangraha/train_split



'Server disconnected without sending a response.' thrown while requesting HEAD https://huggingface.co/datasets/ai4bharat/sangraha/resolve/8b813c3f62d37b2fa174d68c31e8b35ae2fe85e8/sangraha.py
Retrying in 1s [Retry 1/5].
'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/datasets/ai4bharat/sangraha/resolve/8b813c3f62d37b2fa174d68c31e8b35ae2fe85e8/sangraha.py
Retrying in 2s [Retry 2/5].


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/3632345 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

#### Loading the evaluation set

In [6]:
kn_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/kn_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        kn_texts.append(data["text"])

print(f"Successfully read {len(kn_texts)} lines.")   # Should be 1997


en_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/en_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        en_texts.append(data["text"])

print(f"Successfully read {len(en_texts)} lines.")   # Should be 2000 lines

Successfully read 1997 lines.
Successfully read 2000 lines.


#### Loading the new merges list

In [7]:
# Loading the new merges list
final_path = "/Users/nirajh/Code/Language_adaptation/merges_files/new_merges.pkl"

with open(final_path, "rb") as f:
    merges_ = pickle.load(f)

In [8]:
merges_[:5]

[(('à³', 'į'), 574565151),
 (('à³į', 'à²'), 509178184),
 (('à²', '¿'), 471344212),
 (('à²¿', 'à²'), 360436006),
 (('à³', 'ģ'), 323417073)]

#### Loading the Unigram Tokens

In [9]:
# Loading the unigram tokenizer
u_tok = Tokenizer.from_file("/Users/nirajh/Code/Language_adaptation/tokenizers/unigram/unigram_4000.json")

# Getting the original vocabulary of the OLMo tokenizer
orig_vocab = tok.get_vocab()

In [10]:
# Fetching the tokens that are in the Unigram tokenizer's vocab, but are not present in OLMo's original tokenizer
u_new_tokens = ec.unigram_new_tokens(u_tok, orig_vocab)
print("Unigram new Kannada tokens:", len(u_new_tokens))

Unigram new Kannada tokens: 3508


These are the new Kannada tokens that the Unigram tokenizer has learnt. Unigram doesn't learn merges, only tokens (a token that it has learnt could only be a part of a word). The 3,508 tokens we see here will be a part of the model vocabulary (if this tokenizer is used).
<br><br>
<b>Why 3,508 and not 4,000? See below. </b>

In [11]:
# Checking some of these tokens
for t in list(u_new_tokens)[:5]:
    print(t)
    print(ec.to_text(t))
    print()

°à²¾à²ľà³Ģà²µà³į
�ಾಜೀವ್

Ġà²¬à³Ĩà²³à²ķà²¿
 ಬೆಳಕಿ

īà²²à³įà²²à³ĩà²ĸ
�ಲ್ಲೇಖ

°à³ĩà²µà²£à³įà²£
�ೇವಣ್ಣ

Ġà²Ĵà²®à³įà²®à³Ĩ
 ಒಮ್ಮೆ



In [12]:
byte_alpha = set(pre_tokenizers.ByteLevel.alphabet())

def category(p):
    if p in byte_alpha: return "byte alphabet"
    if ec.is_kannada(p): return "Kannada, already in OLMo" if p in orig_vocab else "Kannada, new"
    return "non-Kannada, already in OLMo" if p in orig_vocab else "non-Kannada, new"

print(*Counter(category(p) for p in u_tok.get_vocab()).items(), sep='\n')

('Kannada, new', 3508)
('non-Kannada, new', 16)
('non-Kannada, already in OLMo', 218)
('byte alphabet', 256)
('Kannada, already in OLMo', 2)


The vocabulary size given to the Unigram tokenizer was 4,000. However, we see that it has learnt 3,508 new Kannada tokens. The difference is the above categories. 
- The non-Kannada pieces are for punctuation, digits, and English characters in the training set.
- The Kannada pieces that OLMo already has are à², à³ and Ġà². They contain Kannada lead bytes but aren’t new.
- The 256 byte-level alphabets are from pre_tokenizers.ByteLevel.alphabet(). These are 256 single-character strings, one for every possible byte value (0–255). Each character is the printable stand-in for that byte, using the same table as bytes_to_unicode: for instance Ġ for byte 32 (space) and à for byte 224 (E0). Passing it as initial_alphabet forces the trainer to keep all 256 in the vocabulary, even bytes that never appear in the training text. Any input can then be represented, <b>so there’s never an unknown token</b>.

<b>What are the 218 non-Kannada tokens that are already in OLMo and the 16 non-Kannada new tokens? </b>

In [13]:
def show_category(cat):
    toks = sorted(p for p in u_tok.get_vocab() if category(p) == cat)
    print(f"{cat}: {len(toks)}")
    for p in toks:
        print(f"  {p!r:14} {ec.to_text(p)!r:12} {ec.to_bytes(p).hex(' ')}")

In [14]:
# The following are non-Kannada tokens that were not in OLMo's tokenizer
show_category("non-Kannada, new")

non-Kannada, new: 16
  'Bengaluru'    'Bengaluru'  42 65 6e 67 61 6c 75 72 75
  'Karnataka'    'Karnataka'  4b 61 72 6e 61 74 61 6b 61
  'abha'         'abha'       61 62 68 61
  'annada'       'annada'     61 6e 6e 61 64 61
  'rati'         'rati'       72 61 74 69
  'tory'         'tory'       74 6f 72 79
  'Åĭ'           'ŋ'          c5 8b
  'É¨'           'ɨ'          c9 a8
  'Êī'           'ʉ'          ca 89
  'à°¾à°'        'ా�'         e0 b0 be e0 b0
  'à±ģ'          'ు'          e0 b1 81
  'à±ģà°'        'ు�'         e0 b1 81 e0 b0
  'à±įà°'        '్�'         e0 b1 8d e0 b0
  'êŀĭ'          'Ꞌ'          ea 9e 8b
  'êŀĮ'          'ꞌ'          ea 9e 8c
  'êŀĮa'         'ꞌa'         ea 9e 8c 61


The above category has frequent English words and subword pieces. <br>
It also has some Telugu characters (the bytes `e0 b1` indicate the start of Telugu chars).<br>
Some tokens are phonetic and transliteration letters (`'ŋ' 'ɨ' 'ʉ'`).

In [15]:
# The following tokens were in OLMo's tokenizer, but they're not Kannada characters
show_category("non-Kannada, already in OLMo")

non-Kannada, already in OLMo: 218
  '...'          '...'        2e 2e 2e
  'Do'           'Do'         44 6f
  'India'        'India'      49 6e 64 69 61
  'ST'           'ST'         53 54
  '\\n'          '\\n'        5c 6e
  'ad'           'ad'         61 64
  'aga'          'aga'        61 67 61
  'ago'          'ago'        61 67 6f
  'ai'           'ai'         61 69
  'al'           'al'         61 6c
  'all'          'all'        61 6c 6c
  'am'           'am'         61 6d
  'an'           'an'         61 6e
  'ana'          'ana'        61 6e 61
  'ance'         'ance'       61 6e 63 65
  'any'          'any'        61 6e 79
  'app'          'app'        61 70 70
  'ar'           'ar'         61 72
  'are'          'are'        61 72 65
  'article'      'article'    61 72 74 69 63 6c 65
  'as'           'as'         61 73
  'at'           'at'         61 74
  'ation'        'ation'      61 74 69 6f 6e
  'ba'           'ba'         62 61
  'be'           'be'         62 65
  '

This category has:
- English subword pieces: `form, ent, el, bil, ance`
- Romanized Indian syllables: `ka, ji, aga, ana`
- Devanagari fragments
- Telugu characters
- \n: A backslash followed by the newline character \n
- Mojibake: garbled, unreadable text that occurs when the character encoding that is used to write it is not the same as the one that's used to read it.

Let's check how often these issues occur in the training set:

In [16]:
checks = {
    "Telugu":     re.compile(r"[\u0C00-\u0C7F]"),
    "Devanagari": re.compile(r"[\u0900-\u097F]"),
    "mojibake":   re.compile(r"â€|Ã[\u0080-\u00BF]"),
    "literal \\n": re.compile(r"\\n"),
}

sample = kn_train["train"].shuffle(seed=11).select(range(50_000))["text"]

for name, rx in checks.items():
    n = sum(bool(rx.search(t)) for t in sample)
    print(f"{name:12} {n / len(sample):.2%} of documents")

Telugu       0.10% of documents
Devanagari   0.90% of documents
mojibake     0.00% of documents
literal \n   3.40% of documents


The literal plus newline characters occurs in 3.4% of documents. These can be easily replaced during pretraining. The other issues occur in a very small percentage of documents and don't require any action.

<b>Why we don't have UNK tokens in the Unigram tokenizer's vocabulary: </b><br>
<br>
The 256-byte base alphabet that is a part of the tokenizer's vocabulary is the reason why. Every possible text is a sequence of UTF-8 bytes, and every byte has its own token, so nothing is ever unrepresentable. Text the tokenizer has never seen is broken down into bytes, not characters: an unseen Kannada character becomes up to 3 byte tokens, possibly partly merged, like the prefix + last-byte pairs you saw with the original tokenizer.
<br><br><br>
<b>Do BPE tokenizers not require an UNK token? </b><br>
- Character-level BPEs build merges from Unicode characters. A character never seen in training has no token, so it becomes UNK. This was the original BPE formulation, and SentencePiece by default (SentencePiece has a byte_fallback option that falls back to byte tokens instead). 
- Byte-level BPEs build merges of bytes, so they never need UNK tokens (GPT-2 and OLMo both do this).

#### Initializing function to build the BPE tokenizer

In [17]:
# Getting the JSON version of the regex-modded tokenizer
base_tj = json.loads(kn_tok.to_str())          # regex-modified, no new merges yet
base_tj["normalizer"] = {"type": "NFC"}        # same normalization the merges were learned on

In [18]:
# Initializing function to build a tokenizer with the new list of merges (path to merges list to be given)
def build_tokenizer(n, out_dir, new_merges, model_path):
    
    tkj = copy.deepcopy(base_tj)
    vocab = tkj["model"]["vocab"]
    merge_list = tkj["model"]["merges"]
    as_str = isinstance(merge_list[0], str)     # "a b" or ["a", "b"], depending on version
    next_id = max(max(vocab.values()), max(t["id"] for t in tkj["added_tokens"])) + 1

    for (a, b), _ in new_merges[:n]:
        if a + b not in vocab:                 # reuse the ID if the token already exists
            vocab[a + b] = next_id
            next_id += 1
        merge_list.append(f"{a} {b}" if as_str else [a, b])

    os.makedirs(out_dir, exist_ok=True)
    
    # return AutoTokenizer.from_pretrained(out_dir)
    # The issue with the above line: It doesn't just read tokenizer.json. It also reads tokenizer_config.json.
    # In recent transformers versions, model-specific tokenizer classes can rebuild parts of the pipeline, such as the pre-tokenizer and normalizer, from their own built-in defaults instead of using what’s in tokenizer.json
    # This means the tokenizer is not using the new regex pattern. It is using the pre-built one and fragmenting kannada words.

    hf_tok = PreTrainedTokenizerFast(
        tokenizer_object=Tokenizer.from_str(json.dumps(tkj)),
        eos_token="<|endoftext|>",
        pad_token="<|pad|>",
    )
    hf_tok.save_pretrained(out_dir)     # writes tokenizer.json + a config naming PreTrainedTokenizerFast

    # fail loudly if the pipeline didn't survive the save/load round trip
    reloaded = AutoTokenizer.from_pretrained(out_dir)
    test = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
    pieces = [test[s:e] for _, (s, e) in reloaded.backend_tokenizer.pre_tokenizer.pre_tokenize_str(test)]
    assert pieces == ["ನಮಸ್ಕಾರ", " ಕನ್ನಡ", " ಭಾಷೆಯಲ್ಲಿ"], pieces
    assert reloaded.backend_tokenizer.normalizer is not None, "NFC normalizer was dropped"
    return reloaded



In [19]:
# Building BPE tokenizer
print(f"Number of new merges being used to build the BPE tokenizer: {len(u_new_tokens)}")
bpe_tok = build_tokenizer(len(u_new_tokens), f"/Users/nirajh/Code/Language_adaptation/tokenizers/kn_{len(u_new_tokens)}",
                          merges_, model_dir)

# Getting the new tokens in the BPE tokenizer that are not in the original OLMo model
b_new = ec.bpe_new_tokens(bpe_tok)

Number of new merges being used to build the BPE tokenizer: 3508


In BPE, each merge merge usually creates one new token (unless the result already exists in the vocabulary). Therefore, it is a fair comparison to match `n` Unigram tokens with `n` BPE merges. Since the Unigram tokenizer has learnt 3,508 Kannada tokens, we are using the same number of BPE merges.

<b>The tokenization pipeline </b>

In the tokenizers library, text passes through these stages:
<br>
1. Normalization changes the text itself before anything else. Here, it’s NFC: different Unicode encodings of the same visible text (e.g. a character written as one code point vs. as a base plus a combining mark) are converted to a single standard form. Other tokenizers use normalization for things like lowercasing, removing accents or NFKC. OLMo’s original tokenizer had none; we have added NFC (in this ipynb and tokenizer_edge_cases.py).<br><br>

1. Pre-tokenization splits the text into chunks, roughly words, using regex. It does not produce the pieces. In this pipeline it also includes the ByteLevel step, which converts each chunk to the byte stand-in characters (Ġ, à² and so on). BPE merges and Unigram segmentation never cross the boundaries between chunks.<br><br>

1. The model (BPE or Unigram) splits each chunk into pieces and looks up each piece’s ID in the vocabulary. This is the step the segmenters do (bpe_segmenter and hybrid_segmenter).<br><br>

1. Post-processing adds special tokens, such as a beginning- or end-of-text token, when add_special_tokens=True.

In [20]:
# Getting the base pre tokenizer with the modified regex pattern
pre = kn_tok.pre_tokenizer
segs = {"BPE": ec.cached(ec.bpe_segmenter(bpe_tok)),
        "Unigram": ec.cached(ec.hybrid_segmenter(u_tok, kn_tok.model))}

In [21]:
segs

{'BPE': <function tokenizer_edge_cases.cached.<locals>.f(w)>,
 'Unigram': <function tokenizer_edge_cases.cached.<locals>.f(w)>}

In [22]:
# Reading the word freq file
with open("/Users/nirajh/Code/Language_adaptation/merges_files/word_freq.pkl", "rb") as f:
    train_freq = pickle.load(f)["word_freq"]

In [23]:
train_freq.most_common(5)

[('.', 60116249),
 (',', 41402116),
 ('Ġ', 21435058),
 ('.Ċ', 18937315),
 ('Ġà²Ī', 9952450)]

In [24]:
print(type(kn_texts))
print(type(kn_train))

<class 'list'>
<class 'datasets.dataset_dict.DatasetDict'>


In [25]:
sample = kn_train["train"].shuffle(seed=7).select(range(20_000))
usage_texts = kn_texts + list(sample["text"])
print(len(usage_texts))   # 21,997

21997


In [26]:
b_new_kn = {t for t in b_new if ec.is_kannada(t)}
print(f"BPE new tokens: {len(b_new)}, containing Kannada: {len(b_new_kn)}")
print([ec.to_text(t) for t in b_new - b_new_kn])      # inspect any exceptions

BPE new tokens: 3508, containing Kannada: 3508
[]


### Comparisons

<b>1. Morphology: Suffix bounday rate</b>

In [27]:
# Morphology: boundary at the suffix start (higher = more morpheme-aligned)
morph_res = ec.suffix_boundary_rate(kn_texts, pre, segs)

print("Suffix boundary rate\n")
pd.DataFrame({
    'Suffix': morph_res['BPE'].keys(),
    'BPE': [round(v[0], 5) for v in morph_res['BPE'].values()],
    'Unigram': [round(v[0], 5) for v in morph_res['Unigram'].values()],
    'Count': [v[1] for v in morph_res['Unigram'].values()],
})

Suffix boundary rate



,Suffix,BPE,Unigram,Count
0,ಗಳು,0.48308,0.87569,5792
1,ALL,0.18860,0.46970,67218
2,ನ್ನು,0.10278,0.45989,15782
3,ಗೆ,0.05271,0.27846,16315
4,ಲ್ಲಿ,0.07208,0.24023,15718
5,ಕ್ಕೆ,0.87500,0.91910,4512
6,ಗಳ,0.47329,0.73806,4230
7,ತ್ತಾರೆ,0.00876,0.00584,1027
8,ತ್ತದೆ,0.00061,1.00000,3295
9,ಲ್ಲದೆ,0.34615,0.56538,260


How the suffix boundary rate is calculated:

1. Take every word occurrence in the eval set that ends with one of the listed suffixes (e.g. ನ್ನು, ಲ್ಲಿ, ಗಳು), where the remaining stem is at least 2 characters. If a word matches several suffixes, the longest one counts.
2. Find the position where that suffix begins in the word.
3. Tokenize the word and check whether one of the tokens ends exactly at that position, i.e. whether there's a cut right before the suffix.
4. The rate is the share of those occurrences that have the cut, weighted by frequency.

For example, with ಮರಗಳು and suffix ಗಳು:

ಮರ | ಗಳು → cut before ಗಳು ✓ <br>
ಮ | ರಗಳು → no cut there ✗ <br>
ಮರಗಳು as one token → no cut at all ✗ <br>
ಮರ | ಗ | ಳು → cut before ಗಳು which is further cut to ಗ and ಳು ✓

<b>2. Fertility by frequency range</b>

In [28]:
# Fertility by how common each word was in training
fertility_freq_res = ec.fertility_by_frequency(kn_texts, pre, segs, train_freq)

print("Fertility by frequency range \n")

df_freq = pd.DataFrame.from_dict(fertility_freq_res, orient='index')
df_freq = df_freq.round(5)
df_freq.index.name = 'Frequency range'
display(df_freq)

Fertility by frequency range 



,word occurrences,unique words,BPE,Unigram
Frequency range,,,,
>=10k,295443,10538,1.61022,1.83716
1k-10k,80517,33382,2.89109,3.10203
100-1k,39019,31152,3.38632,3.60809
3-100,26158,23651,4.02749,4.22452
<3 or unseen,8661,8548,5.22503,5.41369


<b>3 + 4. Token utilization </b>

In [29]:
# Dead tokens and utilization (one tokenization pass per tokenizer)
counts = {"BPE": ec.kannada_token_counts(usage_texts, pre, segs["BPE"]),
          "Unigram": ec.kannada_token_counts(usage_texts, pre, segs["Unigram"])}

In [30]:
print("New token usage \n")

df_counts = pd.DataFrame({
    'BPE': dict(ec.new_token_usage(counts["BPE"], b_new)),
    'Unigram': dict(ec.new_token_usage(counts["Unigram"], u_new_tokens))
})

df_counts.index.name = 'Token usage'
display(df_counts.round(3))

New token usage 



,BPE,Unigram
Token usage,,
new tokens,3508.000,3508.000
never used,0.000,2.000
used < 1,0.000,2.000
used < 10,2.000,10.000
used < 100,46.000,23.000
share of Kannada tokens that are new,0.993,0.915


<b>Entropy</b>

In [31]:
df_entropy = pd.DataFrame({
    'BPE': dict(ec.utilization(counts["BPE"])),
    'Unigram': dict(ec.utilization(counts["Unigram"]))
})

display(df_entropy.round(4))

,BPE,Unigram
distinct tokens used,4593.0000,3853.0000
Shannon entropy (bits),10.7298,10.0386
Renyi entropy alpha=2.5 (bits),9.1843,6.8529
Renyi efficiency,0.7550,0.5753


<b>5. Robustness and segmentations </b>

In [32]:
# Robustness and side-by-side segmentations
ec.show_segmentations(ec.ROBUSTNESS_EXAMPLES, pre, segs)

ಮರ ಮರವನ್ನು ಮರದಲ್ಲಿ ಮರದಿಂದ ಮರಗಳು ಮರಗಳಲ್ಲಿ
  BPE      ( 12 tokens): ಮ | ರ |  ಮರ | ವನ್ನು |  ಮರ | ದಲ್ಲಿ |  ಮರ | ದಿಂದ |  ಮರ | ಗಳು |  ಮರ | ಗಳಲ್ಲಿ
  Unigram  ( 13 tokens): ಮ | ರ |  ಮರ | ವನ್ನು |  ಮರ | ದಲ್ಲಿ |  ಮರ | ದಿಂದ |  ಮರ | ಗಳು |  ಮರ | ಗಳ | ಲ್ಲಿ

ಮನೆ ಮನೆಯನ್ನು ಮನೆಯಲ್ಲಿ ಮನೆಗೆ ಮನೆಗಳು
  BPE      (  8 tokens): ಮ | ನೆ |  ಮನ | ೆಯನ್ನು |  ಮನೆಯಲ್ಲಿ |  ಮನೆಗೆ |  ಮನೆ | ಗಳು
  Unigram  (  9 tokens): ಮ | ನೆ |  ಮನೆಯ | ನ್ನು |  ಮನೆ | ಯಲ್ಲಿ |  ಮನೆಗೆ |  ಮನೆ | ಗಳು

ನಾನು ನಾಳೆ office ಗೆ ಹೋಗುತ್ತೇನೆ
  BPE      (  7 tokens): ನಾನು |  ನ | ಾಳೆ |  office |  ಗೆ |  ಹೋಗ | ುತ್ತೇನೆ
  Unigram  (  9 tokens): ನಾ | ನು |  ನಾಳೆ |  office |   | ಗೆ |  ಹೋಗ | ುತ್ತ | ೇನೆ

ನನ್ನ phone number ೯೮೮೦೧೨೩೪೫೬
  BPE      ( 15 tokens): ನ | ನ್ನ |  phone |  number |   | ೯ | ೮ | ೮ | ೦ | ೧ | ೨ | ೩ | ೪ | ೫ | ೬
  Unigram  ( 15 tokens): ನ | ನ್ನ |  phone |  number |   | ೯ | ೮ | ೮ | ೦ | ೧ | ೨ | ೩ | ೪ | ೫ | ೬

ಏನ್ ಮಾಡ್ತಿದೀಯ
  BPE      (  5 tokens): ಏ | ನ್ |  ಮಾಡ್ತ | ಿದ | ೀಯ
  Unigram  (  5 tokens): ಏ | ನ್ |  ಮಾಡ್ತ | ಿದ | ೀಯ

ಕನ್‌ನಡ ಕನ್ನಡ
  BPE      (  6 to

In [33]:
robustness_res = ec.robustness_stats(ec.ROBUSTNESS_EXAMPLES, pre, segs)

df = pd.DataFrame(robustness_res)
df.set_index('text').round(3)

,BPE tokens/word,BPE fragment rate,Unigram tokens/word,Unigram fragment rate
text,,,,
ಮರ ಮರವನ್ನು ಮರದಲ್ಲಿ ಮರದಿಂದ ಮರಗಳು ಮರಗಳಲ್ಲಿ,2.00,0.0,2.167,0.000
ಮನೆ ಮನೆಯನ್ನು ಮನೆಯಲ್ಲಿ ಮನೆಗೆ ಮನೆಗಳು,1.60,0.0,1.800,0.000
ನಾನು ನಾಳೆ office ಗೆ ಹೋಗುತ್ತೇನೆ,1.40,0.0,1.800,0.000
ನನ್ನ phone number ೯೮೮೦೧೨೩೪೫೬,3.75,0.0,3.750,0.000
ಏನ್ ಮಾಡ್ತಿದೀಯ,2.50,0.0,2.500,0.000
ಕನ್‌ನಡ ಕನ್ನಡ,3.00,0.0,3.500,0.286


The segmentations results (especially for the character ಗೆ) show that when the Unigram vocabulary lacks the space-prefixed form of a piece, its most probable segmentation can be the space token (Ġ) followed by the bare piece. Whenever this happens, the word requires two tokens instead of one.

In [34]:
# Checking the number of single-space tokens in the unigram vocabulary
lone_u = counts["Unigram"].get("Ġ", 0) / sum(counts["Unigram"].values())
print(f"Lone-space tokens: {lone_u:.1%} of Unigram's Kannada tokens")

Lone-space tokens: 2.2% of Unigram's Kannada tokens


In [35]:
# Checking the number of single-space tokens in the BPE vocabulary
lone_b = counts["BPE"].get("Ġ", 0) / sum(counts["BPE"].values())
print(f"Lone-space tokens: {lone_b:.1%} of BPE's Kannada tokens")

Lone-space tokens: 0.0% of BPE's Kannada tokens


#### Checking the tokens that are only in Unigram but not in BPE and vice versa

In [36]:
only_unigram = u_new_tokens - b_new
only_bpe = b_new - u_new_tokens
print(f"Shared: {len(u_new_tokens & b_new)}, only Unigram: {len(only_unigram)}, only BPE: {len(only_bpe)}")

Shared: 1419, only Unigram: 2089, only BPE: 2089


In [37]:
def show(tokens, token_counts, k=40):
    for t in sorted(tokens, key=lambda t: -token_counts.get(t, 0))[:k]:
        print(f"{ec.to_text(t)!r:20} {token_counts.get(t, 0):>8,}")

In [38]:
print("\nOnly in the Unigram tokenizer (by Unigram usage):")
show(only_unigram, counts["Unigram"])


Only in the Unigram tokenizer (by Unigram usage):
'ಯಲ್ಲಿ'                48,098
'ತ್ತದೆ'                37,342
' ಪ್ರ�'                36,278
'ಯನ್ನು'                33,912
'�್ದಾರೆ'               26,494
'ಕೊಳ್ಳ'                22,598
' ಸ�'                  19,095
' ನಿ�'                 16,813
'ಯಾ'                   16,543
'ಯು'                   14,981
'ಯೂ'                   14,191
'ರಾ'                   13,691
'ಯೇ'                   13,095
' ಸಾ'                  12,548
'ದ್ದರು'                12,473
'ಯಿಂದ'                 12,023
'�ರ್ಕಾರ'               11,943
' ನಿ'                  11,460
' ಬಾ'                  11,348
'ಕೊಂಡು'                10,766
'ಕಾ'                   10,249
' ಅ�'                  10,246
' ಮಾ'                  10,091
' ಕಾ'                   9,738
' ನಾ'                   9,460
'ವರ್'                   9,189
' ಪಾ'                   9,140
'ರೋ'                    9,084
' ತಿಳಿಸ'                8,947
' ಹಾ'                   8,803
'ಯಾಗ'                   8,380
'ಪಾ'               

Many of these tokens are genuine suffixes and auxiliary verb forms (ಯಿಂದ, ಕೊಂಡು, ತ್ತದೆ, ದ್ದರು, ಕೊಳ್ಳ). The unigram tokenizer's suffix-boundary advantage leads to this.
<br><br>
Many characters are split incorrectly. Eg: ಸ�, �ರ್ಕಾರ, ಪ್ರ�, ನಿ�. These are byte fragments. 

In [39]:
print("\nOnly in the BPE tokenizer (by BPE usage):")
show(only_bpe, counts["BPE"])


Only in the BPE tokenizer (by BPE usage):
'ಗಳನ್ನು'               30,506
' ದ'                   26,684
'ಾನ'                   22,059
'ಾಸ'                   19,384
'ಾವ'                   19,176
' ಲ'                   18,856
'ಿದ್ದಾರೆ'              18,726
' ಯ'                   17,052
'್ವ'                   17,042
'ಗಳಲ್ಲಿ'               16,645
'್ನ'                   16,228
'ಿಯಲ್ಲಿ'               15,382
'ಾಮ'                   15,268
'ಾಲ'                   15,251
'ಿಯನ್ನು'               15,236
'ಿವ'                   14,453
'ಾಕ'                   13,302
' ಟ'                   13,158
'ೆಯಲ್ಲಿ'               13,113
'ೆಯನ್ನು'               12,556
'್ಲ'                   12,527
'ಾತ'                   12,357
'ಿಮ'                   12,181
'ಿದ್ದರು'               11,859
'್ಮ'                   11,737
'್ತ'                   11,711
'್ಕ'                   11,708
'ಿನಲ್ಲಿ'               11,669
'ಿಪ'                   11,430
'ವಾಗಿದೆ'                9,818
'ಿಲ'                    9,788
' ಅನ'                   9,5

<b>Why is the suffix boundary rate calculated? </b>

The suffix boundary rate is being measured to check the frequency of tokens that are vowel-initial tokens. The trained tokenizers (especially the BPE tokenizer), are learning tokens that cross syllable boundaries, such as:<br>
1. <b>Vowel sign plus the next consonant</b>: `ಾನ, ಾಸ, ಾವ, ಾಮ, ಿವ, ಿಮ, ೀರ`. These are the end of one syllable plus the start of the next.
1. <b>A virama plus a consonant</b>: these are the second halves of a conjunct `್ವ, ್ನ, ್ಲ, ್ಮ, ್ತ, ್ಕ, ್ಟ.`
1. <b>Suffix chains that begin with the stem's final vowel sign</b>: ` ಿಯಲ್ಲಿ, ೆಯನ್ನು, ಿದ್ದಾರೆ, ುತ್ತದೆ, ಿನಲ್ಲಿ` the word stem's last consonant gets separated from its own vowel sign. This appears to be the reason for BPE's low suffix-boundary rate.

#### Checking the Unigram tokenizer's fragment rate

In [40]:
bytelevel = decoders.ByteLevel()

def is_kannada(w):
    return "à²" in w or "à³" in w


def hybrid_pieces(text, u):
    text = unicodedata.normalize("NFC", text)
    pieces = []
    for w, _ in kn_tok.pre_tokenizer.pre_tokenize_str(text):
        model = u.model if is_kannada(w) else kn_tok.model
        pieces.extend(t.value for t in model.tokenize(w))
    return pieces


def hybrid_stats(u, texts, lang):
    n_tok = n_words = n_chars = n_frag = 0
    for t in tqdm(texts, leave=False, desc=f"Tokenizer stats on {lang} eval set"):
        pieces = hybrid_pieces(t, u)
        n_tok += len(pieces)
        # n_words += len(t.split())
        # n_chars += len(t)
        n_frag += sum("\ufffd" in bytelevel.decode([p]) for p in pieces)
    return {
        # "fertility (tokens/word)": n_tok / n_words,
        # "chars/token": n_chars / n_tok,
        "fragment rate": n_frag / n_tok,
    }

In [41]:
hybrid_stats(u_tok, kn_texts, "Kannada")

{'fragment rate': 0.1374849090649462}

The fragmentation rate on the Kannada evaluation set for the BPE tokenizer (with 4k new merges) is 0.00518922 (and for 2k new merges, it is 0.00947438).

#### Checking if the Unigram vocab has tokens beginning with split vowel signs

Also checking how many such tokens are present in both tokenizers' vocabs

In [42]:
def starts_with_mark(p):
    t = ec.to_text(p).lstrip(" ")
    return len(t) > 1 and unicodedata.category(t[0]).startswith("M")

In [43]:
for name, vocab, cnt in [("Unigram", u_new_tokens, counts["Unigram"]), ("BPE", b_new, counts["BPE"])]:
    
    marked = sorted((t for t in vocab if starts_with_mark(t)), key=lambda t: -cnt.get(t, 0))
    used = sum(cnt.get(t, 0) for t in marked) / sum(cnt.values())

    print(f"{name}: {len(marked)} mark-initial tokens, {used:.1%} of Kannada token usage")
    print("   ", [ec.to_text(t) for t in marked[:20]])


Unigram: 251 mark-initial tokens, 10.4% of Kannada token usage
    ['ಿದ್ದ', 'ುತ್ತ', 'ಿದೆ', 'ಾರೆ', 'ುವ', 'ಿಗೆ', 'ಿರುವ', 'ಾಗಿದೆ', 'ಿಸಿ', 'ಾಗಿ', 'ುವುದು', 'ಿನ', 'ಿದ್ದು', 'ಾಗ', 'ಿಲ್ಲ', 'ಿಸ', 'ಂತೆ', 'ುವುದ', 'ಿಸಿದ', 'ಿತ್ತು']
BPE: 809 mark-initial tokens, 25.6% of Kannada token usage
    ['ಿಯ', '್ಯ', 'ಿಗೆ', 'ಿನ', 'ಾರ', 'ುವ', 'ೆಯ', 'ಾಗ', 'ಾದ', 'ಿಕ', 'ಿತ', 'ಿದ', 'ಿಸಿ', 'ಿರ', 'ಿದೆ', 'ಿದ್ದ', 'ಾನ', 'ಿಸ', '್ರ', 'ಾಯ']


<b>Issues and next steps for the two tokenizers</b>:

1. <b>Train Unigram tokenizer on characters instead of bytes</b>:
The unigram tokenizer was trained on bytes. This has resulted in fragments of characters instead of whole characters (eg: ಪ್ರ�). <br>Why has this happened?<br> Almost every Kannada character begins with the same two bytes, `E0 B2` or `E0 B3`. A token that ends with those two bytes ( ಪ್ರ + the start of whatever character comes next), fits every word starting with ಪ್ರ, whatever the following character is. So it's far more reusable than any single full-character continuation. To Unigram, that looks just as attractive as a real suffix. But this results in fragmented characters. To prevent this, the Unigram tokenizer should be trained on characters instead of bytes.
<br><br>

1. <b>Exclude merges that begin with a combining mark</b>, i.e., virma (`್`), anusvara (`ಂ` 'um' sound), or vowel sign (` ೆ, ಿ, ು`):
Block any merge whose result is longer than one character and starts with a combining mark (one of the three given above). This should still allow conjuncts (ottaksharas) to form in the right way (`ತ + ್ → ತ್, then ತ್ + ವ → ತ್ವ`). But this should help preserve word roots and prevent them from being separated from their vowel sign (eg: `ಮನೆ` + `ಯಲ್ಲಿ` instead of `ಮನ` + ` ೆಯಲ್ಲಿ`).<br>
The BPE tokenizer has a lot of tokens starting with split-vowel signs (` ೆಯಲ್ಲಿ`) which might be taking up tokens in the vocab and preventing other word roots and suffixes from being learnt. The suffix-boundary rate confirms this: it is 0.1886 for the BPE tokenizer, whereas it is 0.4697 for the Unigram tokenizer. Excluding those merges should help the tokenizer to generalize better.

<b>Why BPE doesn't cut through characters like the Unigram tokenizer did</b>:
<br><br>
BPE builds bottom-up. Its first merges combined bytes to form characters, since these occur the most frequently. When it is looking for new merges, the lead bytes of a Kannada character have already been merged into characters. It simply looks for the next most frequent character combinations, and this is also what leads to tokens that begin with split vowels.
<br><br>
Unigram decides each word's split as a whole, choosing the best overall segmentation, so it doesn't lock in full characters early.